# Step 5D.2e — UNSW-F0: точне порівняння Original (explicit list) vs Audit (algorithmic) pipeline

Мета: перевірити конкретну гіпотезу -- оригінальний Step 5D.2 задає F0 ЯВНИМ списком
(`DATASET_CONFIG["UNSW"]["f0_features"]`), тоді як незалежний аудит (Step 5D.2b)
відновлює F0 АЛГОРИТМІЧНО через `classify_column()`. Обидва дають 41 ознаку -- але
кількість НЕ доводить однаковий ПОРЯДОК колонок, а порядок для LightGBM потенційно
впливає на bin construction / histogram summation order (уже виявлена розбіжність
`Total Bins` у Step 5D.2c/5D.2d).

Це porівняння виконується НА ТІЙ САМІЙ, ОДИН РАЗ завантаженій і розбитій вибірці (той
самий DataFrame, той самий split) -- різниця лише в тому, ЯКИЙ порядок колонок передається
в `X_train`/`X_val`/`X_test` перед `lgb.Dataset`. Це ізолює ЧИСТО ефект порядку колонок від
усіх інших змінних (дані ідентичні за побудовою, а не лише за хешем).

Не чіпаємо frozen experiment, не міняємо split, не міняємо гіперпараметри.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
AUDIT_OUT_DIR = RESULTS_DIR / "step5d2_audit" / "unsw_f0_feature_order_forensic"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR:", RESULTS_DIR, "exists:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR, "exists:", ENRICHED_DIR.exists())


RESULTS_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results exists: True
ENRICHED_DIR: /srv/ids_research/data/enriched exists: True


## 1. ORIGINAL F0 -- буквально скопійований `DATASET_CONFIG["UNSW"]["f0_features"]` з `step5d2_f0_f1_baseline.ipynb`

Це НЕ переказ і не реконструкція за змістом -- це literal copy-paste точного списку, як він
записаний у першоджерелі (клітинка 4 оригінального notebook), символ у символ, включно з
порядком.

In [2]:
ORIGINAL_UNSW_F0 = [
    'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'Sload', 'Dload',
    'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth',
    'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
    'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd',
    'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'proto', 'service',
]
ORIGINAL_UNSW_CATEGORICAL_COLS = ["state", "proto", "service"]  # DATASET_CONFIG["UNSW"]["categorical_cols"]

assert len(ORIGINAL_UNSW_F0) == 41, f"literal copy має {len(ORIGINAL_UNSW_F0)} елементів, очікувалось 41"
print(f"✅ ORIGINAL_UNSW_F0: {len(ORIGINAL_UNSW_F0)} ознак (literal copy з оригінального notebook)")
print(f"   Позиції categorical_cols в ORIGINAL порядку: "
      f"{[(i, c) for i, c in enumerate(ORIGINAL_UNSW_F0) if c in ORIGINAL_UNSW_CATEGORICAL_COLS]}")


✅ ORIGINAL_UNSW_F0: 41 ознак (literal copy з оригінального notebook)
   Позиції categorical_cols в ORIGINAL порядку: [(0, 'state'), (39, 'proto'), (40, 'service')]


## 2. AUDIT F0 -- та сама `classify_column()` логіка, що й у Step 5D.2b/5D.2c (не переписана)

In [3]:
import re

IP_PATTERN = re.compile(r"^(srcip|src_ip|source_ip|dstip|dst_ip|destination_ip|src_addr|dst_addr|srcaddr|dstaddr)$", re.IGNORECASE)
PORT_PATTERN = re.compile(r"^(sport|src_port|source_port|dsport|dst_port|destination_port)$", re.IGNORECASE)
PROTO_PATTERN = re.compile(r"^(proto|protocol)$", re.IGNORECASE)
FLOWID_PATTERN = re.compile(r"(flow.?id|^id$)", re.IGNORECASE)
PROVENANCE_PATTERN = re.compile(r"^(__source_file|__.*)$", re.IGNORECASE)
TIME_COLS_UNSW = {"Stime", "Ltime"}
LABEL_COL_UNSW = "attack_cat"

def classify_column_unsw(col, df):
    if col.startswith("cve_ctx_"):
        return "CTI (cve_ctx_*)"
    if col == LABEL_COL_UNSW:
        return "target"
    if col in TIME_COLS_UNSW:
        return "time"
    if IP_PATTERN.match(col):
        return "network identifier (IP)"
    if PORT_PATTERN.match(col):
        return "network categorical feature (port)"
    if FLOWID_PATTERN.search(col):
        return "identifier (flow ID)"
    if PROVENANCE_PATTERN.match(col):
        return "provenance metadata"
    if col.lower() == "label":
        return "target-derived (redundant)"
    if PROTO_PATTERN.match(col):
        return "predictor (categorical)"
    if col.lower() == "service" and "cve_ctx_service" in df.columns:
        return "predictor (overlap risk)"
    return "predictor"

INCLUDE_ROLES = {"predictor", "predictor (categorical)", "predictor (overlap risk)"}

raw_unsw = pd.read_parquet(ENRICHED_DIR / "unsw_nb15_with_cve_context_dedup.parquet")
roles_unsw = {col: classify_column_unsw(col, raw_unsw) for col in raw_unsw.columns}
AUDIT_UNSW_F0 = [c for c, r in roles_unsw.items() if r in INCLUDE_ROLES]

assert len(AUDIT_UNSW_F0) == 41, f"AUDIT F0 = {len(AUDIT_UNSW_F0)}, очікувалось 41"
print(f"✅ AUDIT_UNSW_F0: {len(AUDIT_UNSW_F0)} ознак (алгоритмічно, classify_column())")


✅ AUDIT_UNSW_F0: 41 ознак (алгоритмічно, classify_column())


## 3. Точне позиційне порівняння ORIGINAL vs AUDIT

Дві перевірки, свідомо РІЗНІ:

1. **Set equality** -- чи це той самий НАБІР ознак (уже опосередковано підтверджено рівністю
   кількості 41=41 у попередніх notebook, тут перевіряється прямо).
2. **Positional equality** -- чи ТОЙ САМИЙ ПОРЯДОК. Це НОВА перевірка, якої не було в
   попередніх notebook -- explicit list і algorithmic reconstruction можуть давати
   однаковий SET при різному ORDER.

In [4]:
set_original = set(ORIGINAL_UNSW_F0)
set_audit = set(AUDIT_UNSW_F0)

print("1. SET EQUALITY:")
if set_original == set_audit:
    print(f"   ✅ Той самий набір ознак ({len(set_original)} елементів)")
else:
    only_orig = set_original - set_audit
    only_audit = set_audit - set_original
    print(f"   ❌ РІЗНІ набори! Лише в ORIGINAL: {only_orig}  Лише в AUDIT: {only_audit}")

print("\n2. POSITIONAL EQUALITY (побудова таблиці для всіх 41 позицій):")
position_rows = []
for i in range(max(len(ORIGINAL_UNSW_F0), len(AUDIT_UNSW_F0))):
    orig_col = ORIGINAL_UNSW_F0[i] if i < len(ORIGINAL_UNSW_F0) else None
    audit_col = AUDIT_UNSW_F0[i] if i < len(AUDIT_UNSW_F0) else None
    match = orig_col == audit_col
    position_rows.append(dict(position=i, original=orig_col, audit=audit_col, match=match))
    marker = "✅" if match else "❌"
    print(f"   {marker} position {i:>2}: original={orig_col!r:<20}  audit={audit_col!r}")

position_df = pd.DataFrame(position_rows)
n_mismatches = int((~position_df["match"]).sum())
print(f"\nПідсумок: {41 - n_mismatches}/41 позицій збігаються точно, {n_mismatches} розходяться.")
if n_mismatches == 0:
    print("✅ ПОРЯДОК КОЛОНОК ІДЕНТИЧНИЙ -- ця гіпотеза (feature order) НЕ пояснює 33 vs 277.")
else:
    print(f"❗ ПОРЯДОК КОЛОНОК РІЗНИТЬСЯ у {n_mismatches} позиціях -- це кандидат-причина, "
          f"яку розділ 5-6 нижче перевірить ЕКСПЕРИМЕНТАЛЬНО (чи впливає це на training).")

position_df.to_csv(AUDIT_OUT_DIR / "unsw_f0_feature_order_comparison.csv", index=False)


1. SET EQUALITY:
   ✅ Той самий набір ознак (41 елементів)

2. POSITIONAL EQUALITY (побудова таблиці для всіх 41 позицій):
   ❌ position  0: original='state'               audit='proto'
   ❌ position  1: original='dur'                 audit='state'
   ❌ position  2: original='sbytes'              audit='dur'
   ❌ position  3: original='dbytes'              audit='sbytes'
   ❌ position  4: original='sttl'                audit='dbytes'
   ❌ position  5: original='dttl'                audit='sttl'
   ❌ position  6: original='sloss'               audit='dttl'
   ❌ position  7: original='dloss'               audit='sloss'
   ❌ position  8: original='Sload'               audit='dloss'
   ❌ position  9: original='Dload'               audit='service'
   ❌ position 10: original='Spkts'               audit='Sload'
   ❌ position 11: original='Dpkts'               audit='Dload'
   ❌ position 12: original='swin'                audit='Spkts'
   ❌ position 13: original='dwin'                audit='Dp

## 4. Побудова ОДНОГО спільного split (та сама сира вибірка), і два ЗРІЗИ з різним порядком колонок

Обидва зрізи (`X_train_original_order`, `X_train_audit_order`) походять з ТОГО САМОГО
`train_df` -- тобто дані побудовані ОДИН раз, а не двома окремими (потенційно
розбіжними) пайплайнами. Це навмисно ізолює ЧИСТО ефект порядку колонок від будь-якої
іншої можливої різниці в data preparation.

In [5]:
df_unsw = raw_unsw.copy()
df_unsw["_is_attack"] = (df_unsw["attack_cat"] != "Normal").astype(int)
df_unsw["ct_ftp_cmd"] = pd.to_numeric(df_unsw["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")

BOUNDARY_1 = pd.Timestamp("2015-02-18 05:36:51")
BOUNDARY_2 = pd.Timestamp("2015-02-18 08:58:36")
t_unsw = pd.to_datetime(df_unsw["Stime"], unit="s")

train_mask = t_unsw < BOUNDARY_1
val_mask = (t_unsw >= BOUNDARY_1) & (t_unsw < BOUNDARY_2)
test_mask = t_unsw >= BOUNDARY_2

train_df = df_unsw[train_mask].copy()
val_df = df_unsw[val_mask].copy()
test_df = df_unsw[test_mask].copy()

assert len(test_df) == 308_935 and int(test_df["_is_attack"].sum()) == 25_354

for col in ["state", "proto", "service"]:
    cats = pd.Categorical(train_df[col]).categories
    train_df[col] = pd.Categorical(train_df[col], categories=cats)
    val_df[col] = pd.Categorical(val_df[col], categories=cats)
    test_df[col] = pd.Categorical(test_df[col], categories=cats)

print(f"✅ Спільний split побудовано: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# два зрізи ТОГО САМОГО DataFrame -- різний лише порядок колонок
X_train_orig, X_val_orig, X_test_orig = train_df[ORIGINAL_UNSW_F0], val_df[ORIGINAL_UNSW_F0], test_df[ORIGINAL_UNSW_F0]
X_train_aud, X_val_aud, X_test_aud = train_df[AUDIT_UNSW_F0], val_df[AUDIT_UNSW_F0], test_df[AUDIT_UNSW_F0]
y_train, y_val, y_test = train_df["_is_attack"], val_df["_is_attack"], test_df["_is_attack"]

print("\nORIGINAL column order (X_train.columns):")
print(" ", list(X_train_orig.columns))
print("\nAUDIT column order (X_train.columns):")
print(" ", list(X_train_aud.columns))


✅ Спільний split побудовано: train=1,441,568  val=308,915  test=308,935

ORIGINAL column order (X_train.columns):
  ['state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'proto', 'service']

AUDIT column order (X_train.columns):
  ['proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd

## 5. Data-identity перевірка (незалежно від порядку) -- per-column hash

Пер-колонна hash (за назвою, а не позицією) МАЄ збігатися для кожної спільної ознаки --
якщо ні, різниця НЕ лише в порядку, а й у самих даних, і це важливіше за порядок.

In [6]:
import hashlib

def hash_series(s):
    return hashlib.sha256(pd.util.hash_pandas_object(s, index=True).values.tobytes()).hexdigest()

mismatched_columns = []
for col in sorted(set_original & set_audit):
    h_orig = hash_series(train_df[col])
    # той самий стовпець з того самого train_df -- hash має бути ІДЕНТИЧНИМ за визначенням,
    # це sanity-check самого способу порівняння, а не пайплайнів
    h_aud = hash_series(train_df[col])
    if h_orig != h_aud:
        mismatched_columns.append(col)

print(f"Sanity-check per-column hash (той самий DataFrame): "
      f"{'✅ 0 розбіжностей (як і очікувалось)' if not mismatched_columns else f'❌ {mismatched_columns}'}")
print(
    "ℹ Це порівняння свідомо тривіальне (той самий DataFrame з собою) -- воно підтверджує "
    "коректність hashing-методу перед його використанням у Step 5D.2f (де порівнюватимуться "
    "ДВА окремо побудовані пайплайни, а не два зрізи одного DataFrame)."
)


Sanity-check per-column hash (той самий DataFrame): ✅ 0 розбіжностей (як і очікувалось)
ℹ Це порівняння свідомо тривіальне (той самий DataFrame з собою) -- воно підтверджує коректність hashing-методу перед його використанням у Step 5D.2f (де порівнюватимуться ДВА окремо побудовані пайплайни, а не два зрізи одного DataFrame).


## 6. Експериментальна перевірка: чи впливає ЧИСТО порядок колонок на training

Тренуємо ДВІ моделі на ТИХ САМИХ даних (той самий `train_df`/`val_df`/`test_df`,
той самий `seed=42`, ті самі гіперпараметри) -- відмінність ЛИШЕ в порядку колонок
(`ORIGINAL` vs `AUDIT` order). Якщо `best_iteration`/`Total Bins`/дерева відрізняються --
порядок колонок ЧАСТКОВО пояснює 33→277 (або принаймні доводить, що LightGBM тут
чутливий до порядку, що само по собі важливо знати для майбутнього frozen protocol).
Якщо ІДЕНТИЧНІ -- ця гіпотеза відхиляється, і треба повертатись до
environment/library-version кандидатів.

In [7]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_STATE = 42
n_pos_train = int(y_train.sum())
n_neg_train = len(y_train) - n_pos_train
SCALE_POS_WEIGHT = n_neg_train / n_pos_train

LGB_PARAMS = dict(
    objective="binary", metric="None", seed=RANDOM_STATE,
    num_leaves=15, learning_rate=0.05, min_data_in_leaf=50,
    scale_pos_weight=SCALE_POS_WEIGHT,
)
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30

def unweighted_average_precision(preds, train_data):
    y_true = train_data.get_label()
    return "ap", average_precision_score(y_true, preds), True


class ListLogger:
    def __init__(self):
        self.lines = []
    def info(self, msg):
        self.lines.append(str(msg))
    def warning(self, msg):
        self.lines.append(str(msg))


def train_with_order(X_train_ord, X_val_ord, X_test_ord, cat_cols):
    logger = ListLogger()
    lgb.register_logger(logger)

    lgb_train = lgb.Dataset(X_train_ord, label=y_train, categorical_feature=cat_cols)
    lgb_val = lgb.Dataset(X_val_ord, label=y_val, reference=lgb_train, categorical_feature=cat_cols)

    model = lgb.train(
        LGB_PARAMS, lgb_train, num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[lgb_val], valid_names=["val"],
        feval=unweighted_average_precision,
        callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                  lgb.log_evaluation(period=0)],
    )
    log_text = "\n".join(logger.lines)
    m = re.search(r"Total Bins (\d+)", log_text)
    total_bins = int(m.group(1)) if m else None

    best_iter = model.best_iteration
    test_scores = model.predict(X_test_ord, num_iteration=best_iter)
    test_ap = average_precision_score(y_test, test_scores)
    return dict(best_iteration=best_iter, test_ap=test_ap, total_bins=total_bins), model


cat_cols_orig = [c for c in ORIGINAL_UNSW_F0 if X_train_orig[c].dtype.name == "category"]
cat_cols_aud = [c for c in AUDIT_UNSW_F0 if X_train_aud[c].dtype.name == "category"]

print("=" * 20, "ORIGINAL порядок колонок", "=" * 20)
result_orig, model_orig = train_with_order(X_train_orig, X_val_orig, X_test_orig, cat_cols_orig)
bi_orig, tap_orig, tb_orig = result_orig["best_iteration"], result_orig["test_ap"], result_orig["total_bins"]
print(f"best_iteration={bi_orig}  test_ap={tap_orig:.6f}  total_bins={tb_orig}")

print("\n" + "=" * 20 + " AUDIT порядок колонок " + "=" * 20)
result_aud, model_aud = train_with_order(X_train_aud, X_val_aud, X_test_aud, cat_cols_aud)
bi_aud, tap_aud, tb_aud = result_aud["best_iteration"], result_aud["test_ap"], result_aud["total_bins"]
print(f"best_iteration={bi_aud}  test_ap={tap_aud:.6f}  total_bins={tb_aud}")

print("\n" + "=" * 60)
print(f"Порівняння: ORIGINAL best_iteration={bi_orig} vs AUDIT best_iteration={bi_aud}")
print(f"Порівняння: ORIGINAL total_bins={tb_orig} vs AUDIT total_bins={tb_aud}")
if result_orig["best_iteration"] == result_aud["best_iteration"] and result_orig["total_bins"] == result_aud["total_bins"]:
    print(
        "✅ Порядок колонок НЕ впливає на best_iteration/Total Bins у цьому середовищі -- "
        "ЦЯ гіпотеза (feature order) ВІДХИЛЯЄТЬСЯ як пояснення 33 vs 277 (Step 5D.2b)."
    )
else:
    print(
        "❗ Порядок колонок ВПЛИВАЄ на best_iteration і/або Total Bins навіть при ІДЕНТИЧНИХ "
        "даних, seed і гіперпараметрах -- це кандидат-причина, яку варто дослідити глибше "
        "(напр. чи саме такий порядок колонок фактично використовувався в оригінальному "
        "Step 5D.2 прогоні, порівняно з тим порядком, який тут використано)."
    )


==================== ORIGINAL порядок колонок ====================
best_iteration=33  test_ap=0.973652  total_bins=5715

==================== AUDIT порядок колонок ====================
best_iteration=277  test_ap=0.982353  total_bins=5715

Порівняння: ORIGINAL best_iteration=33 vs AUDIT best_iteration=277
Порівняння: ORIGINAL total_bins=5715 vs AUDIT total_bins=5715
❗ Порядок колонок ВПЛИВАЄ на best_iteration і/або Total Bins навіть при ІДЕНТИЧНИХ даних, seed і гіперпараметрах -- це кандидат-причина, яку варто дослідити глибше (напр. чи саме такий порядок колонок фактично використовувався в оригінальному Step 5D.2 прогоні, порівняно з тим порядком, який тут використано).


## Явна межа цього forensic-кроку

Обидва варіанти порядку (`ORIGINAL`, `AUDIT`) тренуються в ПОТОЧНОМУ середовищі -- жоден з
них не є тим самим прогоном, що дав історичні `best_iteration=33`. Навіть якщо `ORIGINAL`-
порядок дає інший результат, ніж `AUDIT`-порядок, це показує ЧУТЛИВІСТЬ до порядку колонок,
а не автоматично відновлює число 33 -- для остаточного висновку потрібно порівняти
`result_orig["best_iteration"]` саме з історичним значенням (33), а не лише з `result_aud`.

In [8]:
ORIGINAL_HISTORICAL_BEST_ITERATION = 33
ORIGINAL_HISTORICAL_TEST_AP = 0.973652

bi_orig2, tap_orig2 = result_orig["best_iteration"], result_orig["test_ap"]
print(f"result_orig (ORIGINAL порядок колонок, ПОТОЧНЕ середовище): "
      f"best_iteration={bi_orig2}  test_ap={tap_orig2:.6f}")
print(f"Історичний Step 5D.2 (оригінальне середовище): "
      f"best_iteration={ORIGINAL_HISTORICAL_BEST_ITERATION}  test_ap={ORIGINAL_HISTORICAL_TEST_AP:.6f}")

if result_orig["best_iteration"] == ORIGINAL_HISTORICAL_BEST_ITERATION:
    print(
        "\n🎯 ORIGINAL порядок колонок У ПОТОЧНОМУ СЕРЕДОВИЩІ ВІДТВОРЮЄ історичний "
        "best_iteration=33 -- це було б дуже сильним доказом, що ПОРЯДОК КОЛОНОК Є "
        "справжньою причиною розбіжності Step 5D.2b, а НЕ execution environment/бібліотеки."
    )
else:
    print(
        "\n❌ ORIGINAL порядок колонок У ПОТОЧНОМУ СЕРЕДОВИЩІ НЕ відтворює історичний "
        "best_iteration=33 (навіть якщо він відрізняється від AUDIT-порядку). Це означає, "
        "що порядок колонок -- НЕ єдина причина, і environment/library-version залишаються "
        "провідними кандидатами."
    )


result_orig (ORIGINAL порядок колонок, ПОТОЧНЕ середовище): best_iteration=33  test_ap=0.973652
Історичний Step 5D.2 (оригінальне середовище): best_iteration=33  test_ap=0.973652

🎯 ORIGINAL порядок колонок У ПОТОЧНОМУ СЕРЕДОВИЩІ ВІДТВОРЮЄ історичний best_iteration=33 -- це було б дуже сильним доказом, що ПОРЯДОК КОЛОНОК Є справжньою причиною розбіжності Step 5D.2b, а НЕ execution environment/бібліотеки.


## Збереження артефактів

In [9]:
summary = dict(
    positional_match=bool(n_mismatches == 0),
    n_position_mismatches=int(n_mismatches),
    set_equality=bool(set_original == set_audit),
    result_original_order=dict(
        best_iteration=int(result_orig["best_iteration"]), test_ap=float(result_orig["test_ap"]),
        total_bins=result_orig["total_bins"],
    ),
    result_audit_order=dict(
        best_iteration=int(result_aud["best_iteration"]), test_ap=float(result_aud["test_ap"]),
        total_bins=result_aud["total_bins"],
    ),
    historical_best_iteration=ORIGINAL_HISTORICAL_BEST_ITERATION,
    historical_test_ap=ORIGINAL_HISTORICAL_TEST_AP,
    original_order_matches_historical=bool(result_orig["best_iteration"] == ORIGINAL_HISTORICAL_BEST_ITERATION),
)
with open(AUDIT_OUT_DIR / "unsw_f0_feature_order_forensic_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Збережено в {AUDIT_OUT_DIR}:")
for p in sorted(AUDIT_OUT_DIR.glob("*")):
    print(f"  {p.name}")


Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit/unsw_f0_feature_order_forensic:
  unsw_f0_feature_order_comparison.csv
  unsw_f0_feature_order_forensic_summary.json


## Підсумок

Заповнюється фактичним виводом:

1. Чи `ORIGINAL_UNSW_F0` і `AUDIT_UNSW_F0` позиційно ідентичні (розділ 3).
2. Якщо НІ -- чи навчання з `ORIGINAL`-порядком колонок у поточному середовищі дає інший
   `best_iteration`/`Total Bins`, ніж з `AUDIT`-порядком (розділ 6).
3. Якщо ТАК (різниця є) -- чи `ORIGINAL`-порядок у ПОТОЧНОМУ середовищі відтворює
   історичний `best_iteration=33` (розділ 7) -- це і є вирішальний тест гіпотези.

Якщо порядок колонок виявиться нерелевантним (позиції збігаються АБО навчання з обома
порядками дає однаковий результат), це звужує коло підозрюваних до execution
environment/library version -- саме туди варто повертатись далі.